# Detecção de Defeitos em PCBs utilizando RT-DETR

>&nbsp;&nbsp;&nbsp;&nbsp;Alexandre Augusto Tescaro Oliveira

## 1. Introdução e Motivação

> **Resumo do Estudo:** Este trabalho apresenta o desenvolvimento de um sistema de Visão Computacional para detecção automática de defeitos em Placas de Circuito Impresso (PCBs), utilizando a arquitetura **RT-DETR** (Real-Time DEtection TRansformer). Para viabilizar a execução local e manter comparação consistente entre modelos, o conjunto de dados foi organizado preservando a distribuição das classes de defeito. Os resultados obtidos são comparados com os modelos YOLOv11, Faster R-CNN e RetinaNet.

### 1.1 Contextualização
No cenário da Indústria 4.0, a garantia de qualidade na fabricação de componentes eletrônicos é crítica para reduzir perdas, retrabalho e falhas em campo. As Placas de Circuito Impresso (PCBs) são a base de praticamente todos os dispositivos eletrônicos modernos. Com a miniaturização dos componentes, a inspeção visual tornou-se mais complexa e exige soluções automáticas robustas.

### 1.2 O Problema
Tradicionalmente, a inspeção de PCBs é realizada de forma manual por operadores humanos ou por algoritmos de visão clássica baseados em regras rígidas. Esses métodos apresentam limitações no ambiente industrial:
>* **Fadiga Humana:** A inspeção visual repetitiva aumenta a chance de erro e inconsistências.
>* **Baixa Escalabilidade:** A inspeção manual é lenta e cria gargalos na linha de produção.
>* **Sensibilidade de Regras:** Métodos clássicos falham com variações de iluminação, rotação e ruído.

### 1.3 A Solução Proposta
Para reduzir esses problemas e automatizar o processo de inspeção, este notebook adota Deep Learning com **RT-DETR** (Real-Time DEtection TRansformer), uma arquitetura baseada em Transformers que combina a precisão dos detectores baseados em atenção com a velocidade necessária para aplicações em tempo real.

O objetivo é identificar e localizar seis tipos comuns de defeitos de fabricação:
>1.  **Missing Hole** (Furo faltante)
>2.  **Mouse Bite** (Mordida de rato/Falha na borda)
>3.  **Open Circuit** (Circuito aberto)
>4.  **Short** (Curto-circuito)
>5.  **Spur** (Esporão/Rebarba)
>6.  **Spurious Copper** (Cobre residual)

A aplicação proposta busca aumentar a eficiência do controle de qualidade industrial, reduzindo desperdícios de material e o risco de envio de placas defeituosas.

## 2. Análise Exploratória dos Dados (EDA)
> Notebook pode ser encontrado em ./EDA_VC.ipynb

> Link de acesso ao Dataset utilizado: https://www.kaggle.com/datasets/norbertelter/pcb-defect-dataset

In [1]:
import importlib.util
import subprocess
import sys

FORCE_REINSTALL_TORCH = False
PREFER_CUDA_ON_NVIDIA = True
CUDA_INDEX_URL = "https://download.pytorch.org/whl/cu121"

def _run_cmd(args):
    print("$", " ".join(args))
    subprocess.check_call(args)

def _module_exists(module_name):
    return importlib.util.find_spec(module_name) is not None

def _has_nvidia_gpu():
    try:
        result = subprocess.run(
            ["nvidia-smi", "-L"],
            check=False,
            capture_output=True,
            text=True,
        )
        return result.returncode == 0 and bool(result.stdout.strip())
    except FileNotFoundError:
        return False

def _torch_stack_ready(needs_cuda):
    required_modules = ["torch", "torchvision", "torchaudio"]
    if not all(_module_exists(module_name) for module_name in required_modules):
        return False

    import torch

    if needs_cuda:
        return torch.cuda.is_available() and (torch.version.cuda is not None)
    return True

def _install_torch_stack(needs_cuda):
    install_cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "torch",
        "torchvision",
        "torchaudio",
    ]
    if needs_cuda:
        install_cmd += ["--index-url", CUDA_INDEX_URL]

    try:
        _run_cmd(install_cmd)
    except subprocess.CalledProcessError:
        print("Primeira tentativa falhou. Limpando stack PyTorch e tentando novamente...")
        _run_cmd([sys.executable, "-m", "pip", "uninstall", "-y", "torch", "torchvision", "torchaudio"])
        _run_cmd(install_cmd)

nvidia_gpu_detected = _has_nvidia_gpu()
needs_cuda = PREFER_CUDA_ON_NVIDIA and nvidia_gpu_detected

# Garante ferramentas básicas de build/instalação no venv recém-criado.
_run_cmd([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])

if FORCE_REINSTALL_TORCH or not _torch_stack_ready(needs_cuda):
    target_label = "CUDA 12.1 (cu121)" if needs_cuda else "CPU"
    print(f"Instalando stack PyTorch para {target_label}...")
    _install_torch_stack(needs_cuda)
    print("Stack PyTorch instalada/atualizada.")
else:
    print("Stack PyTorch já compatível com este ambiente.")

required_packages = {
    "pyyaml": "yaml",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "ultralytics": "ultralytics",
    "Pillow": "PIL",
    "numpy": "numpy",
    "certifi": "certifi",
}

missing = [pkg for pkg, module in required_packages.items() if not _module_exists(module)]
if missing:
    _run_cmd([sys.executable, "-m", "pip", "install", "--upgrade", *missing])
    print("Dependências instaladas:", ", ".join(missing))
else:
    print("Dependências já instaladas.")

import torch

print(f"Torch: {torch.__version__} | CUDA build: {torch.version.cuda} | cuda_available={torch.cuda.is_available()}")
if needs_cuda and not torch.cuda.is_available():
    print("ATENÇÃO: GPU NVIDIA detectada, mas CUDA indisponível. Reinicie o kernel e execute novamente esta célula.")

$ /home/alexandre-oliveira/Projetos/pcb-defect-detection/.venv/bin/python -m pip install --upgrade pip setuptools wheel
Stack PyTorch já compatível com este ambiente.
Dependências já instaladas.
Torch: 2.5.1+cu121 | CUDA build: 12.1 | cuda_available=True


In [2]:
# Diagnóstico rápido do runtime PyTorch
import subprocess
import torch

def _has_nvidia_gpu():
    try:
        result = subprocess.run(
            ["nvidia-smi", "-L"],
            check=False,
            capture_output=True,
            text=True,
        )
        return result.returncode == 0 and bool(result.stdout.strip())
    except FileNotFoundError:
        return False

nvidia_gpu_detected = _has_nvidia_gpu()

print(f"GPU NVIDIA detectada: {nvidia_gpu_detected}")
print(f"Torch: {torch.__version__}")
print(f"CUDA build: {torch.version.cuda}")
print(f"cuda_available: {torch.cuda.is_available()}")

if nvidia_gpu_detected and not torch.cuda.is_available():
    print("ATENÇÃO: há GPU NVIDIA, porém CUDA não está ativa. Reexecute a célula anterior e reinicie o kernel.")
elif (not nvidia_gpu_detected) and torch.cuda.is_available():
    print("Observação: CUDA ativa, mas nvidia-smi não foi detectado no PATH.")
else:
    print("Ambiente de execução coerente para seguir com o notebook.")

GPU NVIDIA detectada: True
Torch: 2.5.1+cu121
CUDA build: 12.1
cuda_available: True
Ambiente de execução coerente para seguir com o notebook.


In [3]:
import os
import json
from datetime import datetime
from pathlib import Path

import yaml
import torch
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import RTDETR
from PIL import Image
import numpy as np
import torchvision

# Validação rápida do operador NMS (detecta incompatibilidade torch/torchvision)
try:
    _ = torchvision.ops.nms(
        torch.tensor([[0.0, 0.0, 1.0, 1.0]]),
        torch.tensor([0.9]),
        0.5,
    )
    print("Operador torchvision::nms OK.")
except Exception as exc:
    raise RuntimeError(
        "Falha no operador torchvision."
    ) from exc


Operador torchvision::nms OK.


In [4]:
# Configuração de caminhos
PROJECT_ROOT = Path.cwd()
BASE_DIR = PROJECT_ROOT / "pcb-defect-subset-5000"
RUNS_ROOT = PROJECT_ROOT / "runs" / "detect"

PROJECT_RUN_DIR = RUNS_ROOT / "tcc_pcb_defect_detection" / "rtdetr"
PROJECT_RUN_DIR.mkdir(parents=True, exist_ok=True)

train_images_dir = BASE_DIR / "train" / "images"
val_images_dir = BASE_DIR / "val" / "images"
test_images_dir = BASE_DIR / "test" / "images"

if not train_images_dir.exists():
    raise FileNotFoundError(f"Pasta de treino não encontrada: {train_images_dir}")

data_yaml = {
    "path": str(BASE_DIR),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "names": {
        0: "mouse_bite",
        1: "spur",
        2: "missing_hole",
        3: "short",
        4: "open_circuit",
        5: "spurious_copper",
    },
}

yaml_path = BASE_DIR / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print(f"Dataset base: {BASE_DIR}")
print(f"Arquivo YAML: {yaml_path}")
print(f"Diretório de saída: {PROJECT_RUN_DIR}")

Dataset base: /home/alexandre-oliveira/Projetos/pcb-defect-detection/neural-links-pcb-defect-detection/pcb-defect-subset-5000
Arquivo YAML: /home/alexandre-oliveira/Projetos/pcb-defect-detection/neural-links-pcb-defect-detection/pcb-defect-subset-5000/data.yaml
Diretório de saída: /home/alexandre-oliveira/Projetos/pcb-defect-detection/neural-links-pcb-defect-detection/runs/detect/tcc_pcb_defect_detection/rtdetr


In [5]:
# ==============================================================================
# CHECAR DATA LEAKS
# ==============================================================================

def check_filename_leakage(train_dir, val_dir):
    # Pega apenas os nomes dos arquivos
    train_files = set(os.listdir(train_dir))
    val_files = set(os.listdir(val_dir))

    print(f"Total Treino: {len(train_files)}")
    print(f"Total Validação: {len(val_files)}")

    # Checa duplicatas exatas de nome
    duplicates = train_files.intersection(val_files)
    if duplicates:
        print(f"PERIGO: {len(duplicates)} arquivos têm EXATAMENTE o mesmo nome em Treino e Validação!")
        print(list(duplicates)[:5])
    else:
        print("Nomes de arquivos exatos não se repetem.")

    #  Checa vazamento por Prefixo (Assumindo que o prefixo indica a placa de origem)
    train_prefixes = set([f.split('_')[0] for f in train_files])
    val_prefixes = set([f.split('_')[0] for f in val_files])

    prefix_leak = train_prefixes.intersection(val_prefixes)

    if prefix_leak:
        print(f"ATENÇÃO: {len(prefix_leak)} placas originais (prefixos) aparecem em AMBOS os conjuntos.")
        print(f"Exemplos: {list(prefix_leak)[:5]}")
    else:
        print("Prefixos distintos. Parece que as placas foram separadas corretamente.")


if os.path.exists(train_images_dir) and os.path.exists(val_images_dir):
    check_filename_leakage(train_images_dir, val_images_dir)

Total Treino: 3972
Total Validação: 531
Nomes de arquivos exatos não se repetem.
ATENÇÃO: 3 placas originais (prefixos) aparecem em AMBOS os conjuntos.
Exemplos: ['l', 'light', 'rotation']


## 3. Arquitetura do Modelo: RT-DETR

Neste estudo, o modelo **RT-DETR** (Real-Time DEtection TRansformer) é adotado como detector baseado em Transformers para inspeção automática de PCBs, combinando alta precisão com capacidade de inferência em tempo real.

Diferente dos detectores tradicionais baseados em convoluções (como YOLO ou Faster R-CNN), o RT-DETR utiliza mecanismos de **atenção** (self-attention) para capturar relações globais na imagem, eliminando a necessidade de componentes como NMS (Non-Maximum Suppression) no pós-processamento.

### Por que RT-DETR para PCBs?
A escolha desta arquitetura considera três pontos relevantes para controle de qualidade industrial:

>* **Atenção Global:** O mecanismo de Transformer captura dependências de longo alcance na imagem, ideal para detectar defeitos que dependem do contexto global da placa.
>* **End-to-End:** Eliminação do NMS resulta em pipeline mais simples e previsível, importante para ambientes industriais.
>* **Velocidade em Tempo Real:** Apesar de usar Transformers, o RT-DETR foi otimizado para manter velocidade competitiva com detectores CNN tradicionais.

### Estrutura Simplificada
O fluxo do RT-DETR pode ser resumido nas etapas abaixo:

>1. **Input:** Imagem da PCB é processada para extração de características.
>2. **Backbone (ResNet/HGNetv2):** Extrai mapas de características multiescala.
>3. **Hybrid Encoder:** Combina características intra-escala (AIFI) e cross-escala (CCFM) usando atenção.
>4. **Transformer Decoder:** Processa object queries para predizer diretamente as detecções.
>5. **Saídas finais:** Classe do defeito e *bounding box* refinada, sem necessidade de NMS.

<div align="center">
  <h3>Arquitetura RT-DETR</h3>
  <img src="https://cdn.jsdelivr.net/gh/ultralytics/assets@main/docs/baidu-rtdetr-model-overview.avif" width="760" alt="Diagrama RT-DETR">
</div>

### O Diferencial do RT-DETR: Hybrid Encoder
O RT-DETR introduz um codificador híbrido eficiente que processa características multiescala em dois estágios:

1. **AIFI (Attention-based Intra-scale Feature Interaction):** Aplica self-attention dentro de cada escala para capturar relações espaciais.
2. **CCFM (CNN-based Cross-scale Feature-fusion Module):** Funde informações entre diferentes escalas usando convoluções, mantendo eficiência computacional.

Essa formulação permite ao RT-DETR alcançar precisão superior aos detectores YOLO em muitos benchmarks, mantendo velocidade comparável.

In [6]:
# Treinamento do modelo RT-DETR
if torch.cuda.is_available():
    device_id = 0
    workers = 4
    batch_size = 4  # RT-DETR consome mais memória que YOLO; ajuste conforme sua GPU
    print(f"Executando em CUDA: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device_id = "mps"
    workers = 4
    batch_size = 4
    print("Executando em Apple Silicon MPS")
else:
    device_id = "cpu"
    workers = 2
    batch_size = 2
    print("Executando em CPU")

# Convenção de nomeação:
# AAAAMMDD_HHMM_modelo_img{imgsz}_e{epochs}_bs{batch}_seed{seed}_tag
run_timestamp = datetime.now().strftime("%Y%m%d_%H%M")
run_tag = "baseline"
run_name = f"{run_timestamp}_rtdetr-l_img640_e50_bs{batch_size}_seed42_{run_tag}"

# RT-DETR-l: modelo large com backbone HGNetv2
model = RTDETR("rtdetr-l.pt")
print("Iniciando treinamento RT-DETR")
print(f"Nome do experimento: {run_name}")

# Treinamento com hiperparâmetros consistentes com os outros modelos
results = model.train(
    data=str(yaml_path),
    epochs=100,
    patience=10,
    imgsz=640,
    batch=batch_size,
    project=str(PROJECT_RUN_DIR),
    name=run_name,
    workers=workers,
    lr0=0.0001,  # LR menor para Transformers (recomendação do paper)
    device=device_id,
    verbose=True,

    # REGULARIZAÇÃO
    optimizer='AdamW',
    weight_decay=0.0001,

    # DATA AUGMENTATION FOTOMÉTRICO
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    # DATA AUGMENTATION GEOMÉTRICO
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    flipud=0.5,

    # === Hiperparâmetros Essenciais (não remover) ===
    seed=42,
    deterministic=True,
    amp=False,      # Estabilidade
    cache=False,    # Evitar OOM
    pretrained=True,  # Transfer learning
    val=True,       # Monitoramento de validação
)

results_dir = Path(results.save_dir) if hasattr(results, "save_dir") else Path(str(results))
print(f"Treinamento concluído. Resultados em: {results_dir}")

Executando em CUDA: NVIDIA GeForce GTX 1060 6GB
Iniciando treinamento RT-DETR
Nome do experimento: 20260506_0006_rtdetr-l_img640_e50_bs4_seed42_baseline
Ultralytics 8.4.46 🚀 Python-3.12.3 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1060 6GB, 6070MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/alexandre-oliveira/Projetos/pcb-defect-detection/neural-links-pcb-defect-detection/pcb-defect-subset-5000/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.

/home/alexandre-oliveira/Projetos/pcb-defect-detection/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:825: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/100      5.11G      1.901      1.763     0.5391          4        640: 100% ━━━━━━━━━━━━ 993/993 1.0it/s 16:15<0.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 67/67 2.4it/s 27.7s0.4ss
                   all        531        751   4.38e-05     0.0296   7.34e-06   1.17e-06

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      2/100      2.04G      1.778     0.3724     0.5801          7        640: 0% ──────────── 0/993  1.0s

/home/alexandre-oliveira/Projetos/pcb-defect-detection/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:825: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      2/100      5.44G      1.666     0.4353     0.3976          8        640: 100% ━━━━━━━━━━━━ 993/993 1.2it/s 14:21<0.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 67/67 2.4it/s 27.6s0.4ss
                   all        531        751   1.16e-05     0.0154   1.91e-06   4.02e-07

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      3/100         2G      1.475     0.4655     0.2771         10        640: 0% ──────────── 0/993  1.0s

...

     49/100      5.48G     0.5334     0.4958    0.07971          6        640: 100% ━━━━━━━━━━━━ 993/993 1.2it/s 14:18<0.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 67/67 2.4it/s 27.7s0.4ss
                   all        531        751       0.95      0.963      0.968      0.502

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
     50/100      2.12G     0.5248     0.4584    0.05182          3        640: 0% ──────────── 0/993  1.0s

/home/alexandre-oliveira/Projetos/pcb-defect-detection/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:825: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     50/100      5.44G     0.5242      0.505    0.07848          9        640: 100% ━━━━━━━━━━━━ 993/993 1.2it/s 14:18<0.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 67/67 2.4it/s 27.7s0.4ss
                   all        531        751       0.93      0.965      0.969      0.503

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
     51/100      2.01G     0.4482     0.5992    0.05827         10        640: 0% ──────────── 0/993  1.0s

/home/alexandre-oliveira/Projetos/pcb-defect-detection/.venv/lib/python3.12/site-packages/torch/autograd/graph.py:825: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     51/100      5.47G     0.5196     0.5003    0.07869          4        640: 100% ━━━━━━━━━━━━ 993/993 1.2it/s 14:18<0.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 67/67 2.4it/s 27.7s0.4ss
                   all        531        751       0.93      0.953      0.965        0.5
EarlyStopping: Training stopped early as no improvement observed in last 10 epochs. Best results observed at epoch 41, best model saved as best.pt.
To update EarlyStopping(patience=10) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

51 epochs completed in 12.679 hours.
Optimizer stripped from /home/alexandre-oliveira/Projetos/pcb-defect-detection/neural-links-pcb-defect-detection/runs/detect/tcc_pcb_defect_detection/rtdetr/20260506_0006_rtdetr-l_img640_e50_bs4_seed42_baseline/weights/last.pt, 66.2MB
Optimizer stripped from /home/alexandre-oliveira/Projetos/pcb-defect-detection/neural-links-pcb-defect-

## 4. Análise de Métricas de Desempenho

Para validar a eficácia do RT-DETR na detecção de defeitos em PCBs, utilizamos um conjunto de métricas alinhado ao padrão adotado nos notebooks do YOLO, Faster R-CNN e RetinaNet, com foco em interpretação prática para inspeção industrial.

### Matriz de Confusão
A matriz de confusão mostra, classe por classe, quais defeitos foram corretamente identificados e onde ocorreram confusões.

- **Impacto industrial:** ajuda a identificar erros críticos, como falsos negativos em defeitos que não podem escapar da inspeção.

### Precisão (Precision)
A precisão responde: **"de todos os defeitos que o modelo apontou, quantos eram reais?"**

$$\text{Precision} = \frac{\text{Verdadeiros Positivos}}{\text{Verdadeiros Positivos} + \text{Falsos Positivos}}$$

### Recall (Sensibilidade)
O recall responde: **"de todos os defeitos existentes, quantos o modelo encontrou?"**

$$\text{Recall} = \frac{\text{Verdadeiros Positivos}}{\text{Verdadeiros Positivos} + \text{Falsos Negativos}}$$

### F1-Score
O F1-Score equilibra precisão e recall em um único indicador.

$$F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

### mAP (Mean Average Precision)
- **mAP@50:** indica a qualidade geral de detecção com critério IoU mais permissivo.
- **mAP@50-95:** métrica mais rigorosa, útil para avaliar a qualidade fina de localização das caixas.

In [13]:
print("Rodando validação final no conjunto de teste")
metrics = model.val(split="test")

print("\n--- Métricas de desempenho ---")
print(f"mAP@50: {metrics.box.map50:.4f}")
print(f"mAP@50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

f1_scores = metrics.box.f1
if hasattr(f1_scores, "ndim") and f1_scores.ndim == 1:
    f1_max = f1_scores.max()
else:
    f1_max = np.mean(f1_scores[:, np.argmax(f1_scores.mean(0))])
print(f"F1 máximo: {f1_max:.4f}")

run_dir = Path(model.trainer.save_dir) if hasattr(model, "trainer") and hasattr(model.trainer, "save_dir") else Path(results.save_dir)
csv_path = run_dir / "results.csv"

summary = {
    "map50": float(metrics.box.map50),
    "map50_95": float(metrics.box.map),
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "f1_max": float(f1_max),
    "run_dir": str(run_dir),
}
with open(run_dir / "metrics_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

if csv_path.exists():
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    plt.figure(figsize=(18, 5))

    train_loss_cols = [c for c in df.columns if c.startswith("train/") and "loss" in c]
    val_loss_cols = [c for c in df.columns if c.startswith("val/") and "loss" in c]

    plt.subplot(1, 3, 1)
    if train_loss_cols:
        train_loss = df[train_loss_cols].sum(axis=1)
        plt.plot(df["epoch"], train_loss, label="Treino", linewidth=2)
    if val_loss_cols:
        val_loss = df[val_loss_cols].sum(axis=1)
        plt.plot(df["epoch"], val_loss, label="Validação", linestyle="--", linewidth=2)
    plt.title("Curva de Perda")
    plt.xlabel("Epoch")
    plt.legend()
    plt.grid(True, alpha=0.3)

    map50_col = "metrics/mAP50(B)" if "metrics/mAP50(B)" in df.columns else "metrics/mAP50"
    map95_col = "metrics/mAP50-95(B)" if "metrics/mAP50-95(B)" in df.columns else "metrics/mAP50-95"

    plt.subplot(1, 3, 2)
    plt.plot(df["epoch"], df[map50_col], label="mAP@50", linewidth=2)
    plt.plot(df["epoch"], df[map95_col], label="mAP@50-95", linestyle="--")
    plt.title("mAP")
    plt.xlabel("Epoch")
    plt.legend()
    plt.grid(True, alpha=0.3)

    p_col = "metrics/precision(B)" if "metrics/precision(B)" in df.columns else "metrics/precision"
    r_col = "metrics/recall(B)" if "metrics/recall(B)" in df.columns else "metrics/recall"

    plt.subplot(1, 3, 3)
    plt.plot(df["epoch"], df[p_col], label="Precision")
    plt.plot(df["epoch"], df[r_col], label="Recall")
    plt.title("Precision vs Recall")
    plt.xlabel("Epoch")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    plt.close()
else:
    print("results.csv não encontrado")

cm_paths = [run_dir / "confusion_matrix_normalized.png", run_dir / "confusion_matrix.png"]
for p in cm_paths:
    if p.exists():
        plt.figure(figsize=(8, 8))
        plt.imshow(Image.open(p))
        plt.axis("off")
        plt.title("Matriz de Confusão")
        plt.show()
        plt.close()
        break


Rodando validação final no conjunto de teste
Ultralytics 8.4.46 🚀 Python-3.12.3 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1060 6GB, 6070MiB)


rt-detr-l summary: 310 layers, 31,996,070 parameters, 0 gradients, 103.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3793.5±903.6 MB/s, size: 112.9 KB)
val: Scanning /home/alexandre-oliveira/Projetos/pcb-defect-detection/neural-links-pcb-defect-detection/pcb-defect-subset-5000/test/labels.cache... 391 images, 106 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 497/497 208.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 1.2it/s 26.9s0.8ss
                   all        497        804      0.932      0.953      0.966      0.506
            mouse_bite         56        114      0.914      0.991      0.987      0.529
                  spur         64        130      0.906      0.886      0.933      0.455
          missing_hole         70        136      0.904          1      0.972      0.563
                 short         67        130      0.961      0.941      0.974      0.492
          open_circuit      

<Figure size 1800x500 with 3 Axes>

<Figure size 800x800 with 1 Axes>

## 5. Testes de Inferência

A etapa de inferência foi separada para o notebook `RT-DETR_inferencia.ipynb`, mantendo este arquivo focado em preparação, treinamento e análise de métricas.

Após finalizar um novo treinamento, utilize o melhor checkpoint (`best.pt`) da execução mais recente para validar o comportamento em imagens de teste e em imagens externas.

## 6. Conclusão

O modelo **RT-DETR** foi treinado e avaliado no dataset de defeitos em PCBs, seguindo a mesma metodologia utilizada nos notebooks de YOLOv11, Faster R-CNN e RetinaNet.

Os resultados numéricos devem ser sempre lidos a partir das células de métricas (mAP, precision, recall, F1), pois esses valores variam a cada execução, split e configuração de treino.

Pontos principais observados:

- Na última execução validada, a avaliação apresentou **mAP@50 de 0.9657**, **mAP@50-95 de 0.5057**, **precision de 0.9324**, **recall de 0.9528** e **F1 máximo de 0.9690**.
- **Arquitetura baseada em Transformers:** Utiliza mecanismos de atenção para capturar relações globais na imagem.
- **Detecção End-to-End:** Elimina a necessidade de NMS no pós-processamento, simplificando o pipeline e acelerando a inferência real.
- O desempenho é altamente competitivo com o YOLOv11, mantendo uma abordagem anchor-free robusta para detecção de defeitos pequenos e irregulares.


# 7. Referências  

- Zhao, Y., Lv, W., Xu, S., et al. (2024). DETRs Beat YOLOs on Real-time Object Detection. CVPR 2024.
- Ultralytics RT-DETR Documentation: https://docs.ultralytics.com/models/rtdetr/
- PCB Defect Dataset: https://www.kaggle.com/datasets/norbertelter/pcb-defect-dataset